# Workshop: Build & Deploy a Multi-Agent System on Amazon Bedrock AgentCore

In this workshop you will **build a multi-agent market surveillance system from scratch**, then **deploy it to Amazon Bedrock AgentCore Runtime** - a fully managed container runtime that removes the need to provision or manage servers.

> 📖 **Companion blog post:** [Market Surveillance Agent with LangGraph and Strands on AgentCore](https://aws.amazon.com/blogs/machine-learning/market-surveillance-agent-with-langgraph-and-strands-on-agentcore/) <!-- Update with final URL -->
> This notebook is the hands-on companion to that post. The blog explains the *why* (architecture decisions, the LangGraph + Strands split, the secure tool design); this notebook walks you through the *how*, cell by cell.

The workshop is split into two parts:

| Part | What you will do |
|------|-----------------|
| **Part 1 - Build from scratch** | Create tools, wire up a Strands agent, add an orchestrator, and connect everything with a LangGraph state graph |
| **Part 2 - Scale up & deploy** | Import the full six-agent system and deploy it to AgentCore with one command |

### Architecture

```
                         +--------------------------------------+
                         |  Amazon Bedrock AgentCore Runtime    |
                         |                                      |
  User --> invoke -->    |  LangGraph Workflow                  |
                         |    |- Orchestrator  (routes queries) |
                         |    |- SecurityMonitor (intraday)     |
                         |    |- BrokerMonitor   (multi-day)    |
                         |    |- RiskMonitor     (broker focus) |
                         |    |- IntelAnalyst    (web research) |
                         |    +- Synthesizer     (final report) |
                         +--------------------------------------+
```

**Frameworks used:** [Strands Agents SDK](https://github.com/strands-agents/sdk-python) for individual agents, [LangGraph](https://langchain-ai.github.io/langgraph/) for orchestration, [Amazon Bedrock](https://aws.amazon.com/bedrock/) for model inference, and [AgentCore Runtime](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore.html) for managed deployment.

## Prerequisites

| Requirement | Notes |
|---|---|
| Python 3.11+ | Required by LangGraph and Strands Agents (3.10 works, but 3.11+ is recommended) |
| AWS credentials | `aws sts get-caller-identity` should succeed |
| Bedrock model access | Enable `anthropic.claude-sonnet-4-6` in your target region ([request access](https://console.aws.amazon.com/bedrock/home#/modelaccess)) |
| Docker | Not required — the build runs in the cloud via AWS CodeBuild |

**Required IAM permissions:**

| Service | Actions | Purpose |
|---------|---------|---------|
| `bedrock` | `InvokeModel`, `InvokeModelWithResponseStream`, `ListFoundationModels` | Model inference |
| `ecr` | `CreateRepository`, `GetAuthorizationToken`, `BatchCheckLayerAvailability`, `InitiateLayerUpload`, `UploadLayerPart`, `CompleteLayerUpload`, `PutImage` | Container image registry |
| `codebuild` | `CreateProject`, `StartBuild`, `BatchGetBuilds` | ARM64 image build |
| `iam` | `CreateRole`, `AttachRolePolicy`, `PutRolePolicy`, `PassRole` | Execution role for the runtime |
| `bedrock-agentcore` | `CreateMemory`, `GetMemory`, `ListMemories`, `CreateAgentRuntime`, `InvokeAgentRuntime`, `GetAgentRuntimeEndpoint`, `DeleteAgentRuntime` | Memory + managed runtime lifecycle |
| `s3` | `CreateBucket`, `PutObject`, `GetObject` | CodeBuild source bucket |

> **Cost notice:** Running this notebook creates billable AWS resources (CodeBuild, ECR, AgentCore Memory, an AgentCore Runtime endpoint, and Bedrock inference on every invocation). Run the **Cleanup** section at the end to stop ongoing charges.

---
## 1. Setup

In [ ]:
# Install dependencies
%pip install -r ../requirements.txt

In [ ]:
import boto3
import json
from pathlib import Path
from boto3.session import Session
import sys

# The notebook lives in notebooks/, but `src` imports (Part 2) and the
# deployment entrypoint assume the repository root. Resolve it once and put
# it on sys.path so `from src...` works regardless of kernel start dir.
REPO_ROOT = Path.cwd() if (Path.cwd() / "api.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


# ---- Region & model configuration (edit these) -----------------------------
# Notebook uses Claude Sonnet 4.6; The model is reached through cross-region inference
# Set REGION to match, or change the model IDs to the prefix for your region.
REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-sonnet-4-6"


boto_session = Session()
region = REGION or boto_session.region_name or "us-east-1"

sts = boto3.client("sts", region_name=region)
identity = sts.get_caller_identity()

print(f"Region        : {region}")
print(f"Model  : {MODEL_ID}")


---
# Part 1 - Build a Surveillance Agent from Scratch

We start with one narrowly scoped agent. In this part you will:
1. Define a mock market-data layer
2. Wrap it with Strands `@tool` functions
3. Create a single surveillance agent that uses the tools
4. Add an orchestrator agent for routing
5. Wire everything together with a LangGraph state graph

## 2. Define the Data Layer

The sample uses four in-memory CSV tables. A real surveillance system would put the same contract in front of a database, data warehouse, or API.

The catalog separates two concepts:

- `AGENT_REPORT_ACCESS` describes which reports each specialist should discover.
- `REPORT_SCHEMAS` defines the columns that may be used as filters.

The catalog helps the model select data, but it is not an authorization boundary. Production access control must be enforced by the data service or by agent-scoped tool credentials.

In [ ]:
# - Mock market data --------------------------------------------
# In production these would be database tables.  The agent doesn't
# know the data is fake - it discovers the schema via tools.

MOCK_DATA = {
    "TradeActivity": (
        "date,time,symbol,price,volume,side,broker\n"
        "2024-03-15,09:35:00,AAPL,172.50,1500,BUY,ALPHA_CAPITAL\n"
        "2024-03-15,09:35:02,AAPL,172.55,800,BUY,SUMMIT_TRADING\n"
        "2024-03-15,09:36:15,AAPL,172.30,2200,SELL,VERTEX_SECURITIES\n"
        "2024-03-15,10:00:05,AAPL,173.00,2500,BUY,SUMMIT_TRADING\n"
        "2024-03-15,10:15:00,AAPL,173.10,1100,SELL,CRESTVIEW_CAPITAL\n"
        "2024-03-15,10:30:00,AAPL,173.25,3200,BUY,PINNACLE_PARTNERS\n"
        "2024-03-15,09:40:10,MSFT,425.20,3000,BUY,PINNACLE_PARTNERS\n"
        "2024-03-15,10:05:30,MSFT,425.50,4000,BUY,VERTEX_SECURITIES\n"
        "2024-03-15,10:20:45,MSFT,425.80,2800,BUY,ALPHA_CAPITAL"
    ),
    "PriceBars": (
        "date,time,symbol,high,low,volume,num_trades,vwap\n"
        "2024-03-15,09:30:00,AAPL,172.80,171.90,15000,45,172.35\n"
        "2024-03-15,10:00:00,AAPL,173.20,172.10,22000,67,172.65\n"
        "2024-03-15,10:30:00,AAPL,173.50,172.80,18000,52,173.15\n"
        "2024-03-15,11:00:00,AAPL,173.80,173.20,14000,41,173.50\n"
        "2024-03-15,09:30:00,MSFT,425.50,424.80,12000,38,425.15\n"
        "2024-03-15,10:00:00,MSFT,426.00,425.20,19000,55,425.60"
    ),
}

# - Report catalog ----------------------------------------------
# Agents call get_report_list() to discover what data is available.

AVAILABLE_REPORTS = [
    {
        "report_name": "TradeActivity",
        "description": "Tick-level trade data with price, volume, side, and executing broker.",
    },
    {
        "report_name": "PriceBars",
        "description": "Aggregated price bars with high, low, volume, num_trades, and VWAP.",
    },
]

# - Report schemas ----------------------------------------------
# Agents call get_report_schema() to learn column names before
# building SQL queries.

REPORT_SCHEMAS = {
    "TradeActivity": {
        "displayName": "Trades",
        "reportName": "TradeActivity",
        "parameters": [
            {"name": "date", "description": "Trading date (YYYY-MM-DD)", "required": True},
            {"name": "symbol", "description": "Stock ticker", "required": False},
        ],
        "columns": [
            {"name": "date"},  {"name": "time"},   {"name": "symbol"},
            {"name": "price"}, {"name": "volume"}, {"name": "side"},
            {"name": "broker"},
        ],
    },
    "PriceBars": {
        "displayName": "Price Bars",
        "reportName": "PriceBars",
        "parameters": [
            {"name": "date", "description": "Trading date (YYYY-MM-DD)", "required": True},
            {"name": "symbol", "description": "Stock ticker", "required": True},
        ],
        "columns": [
            {"name": "date"},       {"name": "time"},   {"name": "symbol"},
            {"name": "high"},       {"name": "low"},    {"name": "volume"},
            {"name": "num_trades"}, {"name": "vwap"},
        ],
    },
}

print(f"Loaded {len(MOCK_DATA)} tables, {sum(d.count(chr(10)) for d in MOCK_DATA.values())} rows total")

## 3. Define Agent Tools

Strands agents discover their capabilities through **tools** - Python functions decorated with `@tool`. Each tool's docstring is what the LLM reads to decide when and how to call it.

Our agents use a three-step pipeline:

```
get_report_list  ->  get_report_schema  ->  run_report
  "what data          "what columns        "filter + fetch the rows
   exists?"            does it have?"        (safe, bound parameters)"
```

Notice there is **no** "write SQL" tool. The LLM picks a report and supplies *filter values* (e.g. `{"symbol": "AAPL"}`); `run_report` validates those fields against the report's schema and binds the values as parameters. The model never emits raw SQL as a guardrail for preventing SQL-injections

In [ ]:
import json
import csv
import io
from typing import Dict, Optional
from strands import tool


@tool
def get_report_list() -> str:
    """Get the list of available market data reports with descriptions.

    Returns:
        str: JSON array of report objects with name and description.
    """
    return json.dumps(AVAILABLE_REPORTS, indent=2)


@tool
def get_report_schema(report_name: str) -> str:
    """Get the column schema for a report so you can choose filter fields.

    Args:
        report_name: Name of the report (e.g. 'TradeActivity', 'PriceBars').

    Returns:
        str: JSON object with parameter and column definitions.
    """
    schema = REPORT_SCHEMAS.get(report_name)
    if not schema:
        raise ValueError(f"Unknown report: {report_name}. Available: {list(REPORT_SCHEMAS.keys())}")
    return json.dumps(schema, indent=2)


@tool
def run_report(report_name: str, filters: Dict[str, str], limit: Optional[int] = None) -> dict:
    """Run a predefined report with schema-validated, parameter-bound filters.

    Args:
        report_name: A report from get_report_list (e.g. 'TradeActivity').
        filters: Equality filters keyed by column, e.g.
                 {"symbol": "AAPL", "date": "2024-03-15"}.
        limit: Optional row cap (1..10000).

    Returns:
        dict: {'success': bool, 'data': str (CSV), 'error': str or None, 'sql': str}
    """
    schema = REPORT_SCHEMAS.get(report_name)
    if not schema:
        return {"success": False, "data": "", "error": f"Unknown report: {report_name}"}

    # 1) Allowlist - reject any filter field that is not a column of this report.
    allowed = {c["name"] for c in schema["columns"]}
    unknown = set(filters) - allowed
    if unknown:
        return {"success": False, "data": "",
                "error": f"Unknown filter field(s) {sorted(unknown)}. Allowed: {sorted(allowed)}"}

    # 2) Build SQL with NAMED BIND PARAMETERS (:symbol, :date)
    where = " AND ".join(f"{field} = :{field}" for field in filters)
    sql = f"SELECT * FROM {schema['reportName']}"
    if where:
        sql += f" WHERE {where}"
    if limit is not None:
        if not isinstance(limit, int) or not 1 <= limit <= 10_000:
            return {"success": False, "data": "", "error": "limit must be an integer in [1, 10000]"}
        sql += f" LIMIT {limit}"

    # 3) Execute against the mock data, treating the bound values as literals.
    #    In a real scenario you would hand (sql, filters) to a parameterised driver
    #    (SQLAlchemy text().bindparams, psycopg, or Athena ExecutionParameters).
    rows = list(csv.DictReader(io.StringIO(MOCK_DATA[report_name])))
    matched = [r for r in rows if all(str(r.get(k)) == str(v) for k, v in filters.items())]
    if limit is not None:
        matched = matched[:limit]

    out = io.StringIO()
    fieldnames = list(rows[0].keys()) if rows else []
    writer = csv.DictWriter(out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(matched)
    return {"success": True, "data": out.getvalue(), "error": None, "sql": sql}


print("Tools defined: get_report_list, get_report_schema, run_report")

## 4. Your First Surveillance Agent

Now we give these tools to a **Strands Agent** backed by Claude Sonnet 4.6 on Bedrock. The agent is defined by its system prompt and the tools it can access. It decides which tools to call and chains them together to answer the user's question.

We enable **thinking** so the agent reasons step-by-step before and between tool calls - this improves accuracy on analytical tasks.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from botocore.config import Config

# Sonnet 4.6 uses *adaptive* thinking: the model decides when and how much to think, and `effort` tunes the depth.
analyst_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    max_tokens=16000,
    boto_client_config=Config(
        retries={"max_attempts": 3, "mode": "adaptive"},
        read_timeout=300,
    ),
    additional_request_fields={
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": "high"},
    },
)

# Faster model config: used for routing and synthesis. We disable thinking where we
# want speed and deterministic output over deep reasoning.
fast_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    temperature=0.0,
    max_tokens=4000,
)

print(f"Models configured (region={region}, model={MODEL_ID})")


### Define the system prompt

The system prompt tells the agent what it is, what tools it has, and how to use them. The three-step tool pipeline is spelled out so the agent follows it consistently.

In [ ]:
ANALYST_PROMPT = '''You are a Security Monitor Agent that analyses single-day market data.

## Workflow
1. Call get_report_list to discover what data is available.
2. Pick the right report for the question and call get_report_schema to see its columns.
3. Call run_report with the report name and your filters
   (e.g. {"symbol": "AAPL", "date": "2024-03-15"}).
4. Analyse the rows it returns and present findings with specific numbers.

## Output Format
{
  "objective": "What the user asked",
  "business_insights": "Your findings backed by data"
}

Stay within the available data. If the data cannot answer the question, say so.'''

# Create the agent - Strands wires the tools, model, and prompt together
analyst = Agent(
    model=analyst_model,
    system_prompt=ANALYST_PROMPT,
    tools=[get_report_list, get_report_schema, run_report],
)

print("Analyst agent created")

### Run the agent

Ask a question and watch the agent reason through its tool pipeline. The default callback handler prints tool calls and text as they happen.

In [ ]:
result = analyst("What trades happened on AAPL on March 15, 2024?")

### Stream Agent Output

The cell above used the **default callback handler**, which prints as the agent works. For a real UI you would need the raw event stream instead. `stream_async` yields typed events - text chunks, tool calls, and reasoning tokens - as they happen.

We iterate those events by hand below. This is the exact pattern each LangGraph node will reuse to forward agent events into the graph's stream (Section 6).

In [ ]:
# A fresh agent with the default printing suppressed (callback_handler=None),
# so we can handle the event stream ourselves.
analyst_streaming = Agent(
    model=analyst_model,
    system_prompt=ANALYST_PROMPT,
    tools=[get_report_list, get_report_schema, run_report],
    callback_handler=None,
)

previous_tool = None
async for event in analyst_streaming.stream_async("Show me the price bars for AAPL on March 15, 2024"):
    if "data" in event:
        print(event["data"], end="", flush=True)
    elif "reasoningText" in event:
        pass  # hide thinking tokens for cleaner output
    elif event.get("current_tool_use", {}).get("name"):
        tool_name = event["current_tool_use"]["name"]
        if tool_name != previous_tool:
            print(f"\n  >> tool: {tool_name}")
            previous_tool = tool_name

## 5. Add an Orchestrator

A single agent works for one kind of question, but real workloads are mixed. We add an **orchestrator** that reads each query and decides which specialist(s) should handle it.

We do this with an explicit routing node rather than the "agents as tools" pattern (where each specialist is wrapped as a tool the model can call). For surveillance work, explicit routing provides three advantages:

- **Controlled flow** - we are in control of the orchestration and flow.
- **Context management** - results go to typed state consumed by the synthesizer, rather than sub-agent outputs returning directly to the main agent.
- **Durable execution** — LangGraph checkpoints save the state after every node allowing to recover from errors without re-running the full pipeline.

Our orchestrator has no tools; it just emits a JSON routing decision that the graph acts on.

In [ ]:
ORCHESTRATOR_PROMPT = '''You are an orchestrator for a market surveillance system.
Analyse the user query and decide whether it needs data analysis.

If YES, respond with ONLY this JSON (no other text):
{
  "agent_tasks": [
    {"agent": "analyst", "task": "specific analysis task description"}
  ]
}

If NO (e.g. a greeting or general knowledge question), respond conversationally.

RULES:
- One task per agent. Keep it focused.
- Include specific dates and securities mentioned in the query.
- If a required security code is missing, ask the user for it.'''

orchestrator = Agent(
    model=fast_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[],  # no tools - the orchestrator only routes
    callback_handler=None,
)

# this should produce a JSON routing decision
result = orchestrator("What was the trading volume of AAPL on March 15, 2024?")
print("--- Routing decision ---")
print(result)

## 6. Wire It Together with LangGraph

As the complexity of your solution increases, with more possible routes and sub-agents, it becomes harder to maintain and manage.

**LangGraph** addresses this by modeling complex workflows as directed graphs. Each agent becomes a node; edges describe the allowed transitions; a shared `State` flows between them. The orchestrator decides the route and conditional edges pick the next node. The key integration of our Strands agent lives inside each node.

### Add short-term memory with AgentCore Memory

Before we compile the graph, we create the **memory** it will checkpoint into. We use **Amazon Bedrock AgentCore Memory** for short-term state: `AgentCoreMemorySaver` (from `langgraph-checkpoint-aws`) is a LangGraph checkpointer that saves the graph state after every node. Conversations therefore survive restarts and can be resumed by `thread_id`.

Creating a memory resource takes a minute or two. It's a one time setup, run the cell below and wait for AgentCore Memory resource to become `ACTIVE`.

In [ ]:
# - Create an AgentCore Memory resource -------------------------
import time

control_client = boto3.client("bedrock-agentcore-control", region_name=region)

MEMORY_NAME = "WorkshopSurveillanceMemory"
MEMORY_ID = None

try:
    memory_response = control_client.create_memory(
        name=MEMORY_NAME,
        description="Short-term memory for the workshop LangGraph workflow.",
        eventExpiryDuration=7,  # days
    )
    MEMORY_ID = memory_response["memory"]["id"]
    print(f"Created AgentCore Memory: {MEMORY_ID}")
except control_client.exceptions.ValidationException:
    # A memory with this name already exists (the name is encoded in its id).
    # Reuse it instead of failing - for subsequent notebook runs.
    for mem in control_client.list_memories().get("memories", []):
        if MEMORY_NAME in (mem.get("id") or "") or MEMORY_NAME in (mem.get("name") or ""):
            MEMORY_ID = mem["id"]
            print(f"Using existing AgentCore Memory: {MEMORY_ID}")
            break

if MEMORY_ID is None:
    raise RuntimeError(f"Could not create or find memory {MEMORY_NAME!r}")

# Wait for ACTIVE, with a 10-minute deadline. Creation normally takes 1-3 min;
# the deadline stops the loop hanging on a stuck or FAILED resource.
deadline = time.time() + 600
while True:
    status = control_client.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if status == "ACTIVE":
        break
    if status == "FAILED" or time.time() >= deadline:
        raise RuntimeError(f"Memory {MEMORY_ID} is {status!r} (expected ACTIVE)")
    time.sleep(10)
print("Memory is ACTIVE")

### Build the graph

Now we define the shared state, the three async nodes (orchestrator, analyst, synthesizer), the routing function, and the edges - then compile the graph using the memory resource above as its checkpointer. Each node returns **only the fields it sets**; LangGraph merges those partial updates into the shared state.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import END, StateGraph
from langgraph.config import get_stream_writer
from langgraph_checkpoint_aws import AgentCoreMemorySaver


# - State schema ------------------------------------------------
# The shared state flows through every node. total=False lets each node return
# only the keys it sets (a partial update), which LangGraph merges in.
class SurveillanceState(TypedDict, total=False):
    query: str
    needs_analysis: Optional[bool]
    agent_task: Optional[str]
    analyst_result: Optional[str]
    final_response: Optional[str]


# - Node: Orchestrator ------------------------------------------
async def orchestrator_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "agent_execution", "agent": "orchestrator", "status": "started"})

    orch = Agent(model=fast_model, system_prompt=ORCHESTRATOR_PROMPT, tools=[], callback_handler=None)
    chunks = []
    async for event in orch.stream_async(state["query"]):
        if "data" in event:
            writer({"type": "text", "content": event["data"], "agent": "orchestrator"})
            chunks.append(event["data"])

    response = "".join(chunks)
    writer({"type": "agent_execution", "agent": "orchestrator", "status": "completed"})

    # Parse the routing decision. The model may return a bare JSON object, a
    # fenced block, or JSON wrapped in prose - so we strip an optional code
    # fence and, if needed, fall back to the first JSON object in the text.
    needs_analysis, agent_task = False, None
    candidate = response.strip()
    if candidate.startswith("```"):
        newline = candidate.find("\n")
        if newline != -1:
            candidate = candidate[newline + 1:]
        if candidate.rstrip().endswith("```"):
            candidate = candidate.rstrip()[:-3]
    candidate = candidate.strip()
    try:
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            start = candidate.find("{")
            if start < 0:
                raise
            parsed, _ = json.JSONDecoder().raw_decode(candidate[start:])
        tasks = parsed.get("agent_tasks", [])
        if tasks:
            needs_analysis = True
            agent_task = tasks[0]["task"]
    except (json.JSONDecodeError, IndexError, KeyError, AttributeError):
        pass

    # Return ONLY the fields this node sets - LangGraph merges partial updates.
    return {"needs_analysis": needs_analysis, "agent_task": agent_task}


# - Node: Analyst -----------------------------------------------
async def analyst_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "agent_execution", "agent": "analyst", "status": "started"})

    agent = Agent(
        model=analyst_model,
        system_prompt=ANALYST_PROMPT,
        tools=[get_report_list, get_report_schema, run_report],
        callback_handler=None,
    )

    prompt = state.get("agent_task") or state["query"]
    chunks, prev_tool = [], None
    async for event in agent.stream_async(prompt):
        if "data" in event:
            writer({"type": "text", "content": event["data"], "agent": "analyst"})
            chunks.append(event["data"])
        elif event.get("current_tool_use", {}).get("name"):
            tool_name = event["current_tool_use"]["name"]
            if tool_name != prev_tool:
                writer({"type": "tool", "name": tool_name, "agent": "analyst", "status": "started"})
                prev_tool = tool_name

    writer({"type": "agent_execution", "agent": "analyst", "status": "completed"})
    return {"analyst_result": "".join(chunks)}


# - Node: Synthesizer ------------------------------------------
async def synthesizer_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "agent_execution", "agent": "synthesizer", "status": "started"})

    synth = Agent(
        model=fast_model,
        system_prompt="Synthesise analyst findings into a clear, structured summary with bullet points.",
        tools=[],
        callback_handler=None,
    )

    prompt = f"Original question: {state['query']}\n\nAnalyst findings:\n{state.get('analyst_result', '')}"
    chunks = []
    async for event in synth.stream_async(prompt):
        if "data" in event:
            writer({"type": "text", "content": event["data"], "agent": "synthesizer"})
            chunks.append(event["data"])

    writer({"type": "agent_execution", "agent": "synthesizer", "status": "completed"})
    return {"final_response": "".join(chunks)}


# - Routing function --------------------------------------------
def should_analyse(state: SurveillanceState) -> str:
    return "analyst" if state.get("needs_analysis") else END


# - Build the graph ---------------------------------------------
workflow = StateGraph(SurveillanceState)
workflow.add_node("orchestrator", orchestrator_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.set_entry_point("orchestrator")
workflow.add_conditional_edges("orchestrator", should_analyse, {"analyst": "analyst", END: END})
workflow.add_edge("analyst", "synthesizer")
workflow.add_edge("synthesizer", END)

# Use the AgentCore Memory resource (created above) as the checkpointer.
checkpointer = AgentCoreMemorySaver(MEMORY_ID, region_name=region)
graph = workflow.compile(checkpointer=checkpointer)
print("LangGraph workflow compiled with AgentCore Memory checkpointer")

### Run the Workflow

`graph.astream(..., stream_mode="custom")` delivers every `writer()` call from inside the nodes as a real-time chunk. This is exactly how the Streamlit UI and the AgentCore SSE endpoint stream output later - the same events (`agent_execution`, `text`, `tool`) are surfaced to the frontend. We pass a `thread_id` as our session identifier, so the checkpointer can persist and resume this conversation.

In [ ]:
async for chunk in graph.astream(
    {"query": "What was the trading volume for AAPL on March 15, 2024?"},
    config={"configurable": {"thread_id": "workshop-001", "actor_id": "workshop-user"}},
    stream_mode="custom",
):
    ctype = chunk.get("type", "")
    agent = chunk.get("agent", "")

    if ctype == "agent_execution":
        icon = ">" if chunk["status"] == "started" else "ok"
        print(f"\n{icon} {agent}")
    elif ctype == "text":
        print(chunk["content"], end="", flush=True)
    elif ctype == "tool":
        print(f"\n  [tool] {chunk['name']}")

> **What just happened?** The orchestrator parsed the query, decided it needed analysis, and routed to the analyst. The analyst called three tools in sequence (`get_report_list` -> `get_report_schema` -> `run_report`), analysed the rows, and passed its findings to the synthesizer for a final summary. Every step streamed in real time through LangGraph.

You now have a working multi-agent workflow. In Part 2 we scale this to the full six-agent system and deploy it to AgentCore.

---
# Part 2 - Scale Up & Deploy to AgentCore

The workflow you just built demonstrates the core patterns. 

Part 2 showcases a comprehensive multi-agent solution, with more agents, more data and managed runtime deployment:

| Feature | Part 1 | Part 2 |
|---------|--------------|---------------------|
| Specialist agents | 1 (security) | 4 (security, broker, risk, intel) |
| Routing | Dynamic: walks `required_agents` by index | Dynamic: any combination of agents |
| Data tables | 2 | 7 (trades, price bars, broker activity, risk scores, order book, daily snapshots, volume profile) |
| Checkpointer | AgentCore Memory | AgentCore Memory |
| Deployment | Local | AgentCore Runtime |

## 7. The Full Multi-Agent Workflow

Let's add more specialist agents. Real surveillance work doesn't happen in isolation. Analysts cross-reference intraday activity against historical patterns, check who was behind the trades, and pull in external news before drawing conclusions. It's the combination that turns data points into an informed view.

We capture that by adding more specialists, each with its own process and data it looks at:


**Security Monitor** - Single-day intraday activity — price, volume, tick-level trades

**Broker Monitor** - Multi-day trends: historical prices, volatility, risk score evolution

**Risk Monitor** - Broker behaviour on a single day: execution patterns, market share, anomaly flags

**Intel Analyst**: news, filings, public disclosures

The orchestrator picks any combination of these based on the query, and a synthesizer funnels their findings into one comprehensive report.

![Multi-agent orchestration graph](../langgraph-architecture.svg)


In [ ]:
import json
from typing import Any, TypedDict
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph
from strands_tools import http_request

#Import agent system prompts
from src.prompts.system_prompts import (
    BROKER_MONITOR_AGENT_PROMPT,
    INTEL_ANALYST_AGENT_PROMPT,
    ORCHESTRATOR_AGENT_PROMPT,
    RISK_MONITOR_AGENT_PROMPT,
    SECURITY_MONITOR_AGENT_PROMPT,
)

SPECIALISTS = (
    "security_monitor",
    "broker_monitor",
    "risk_monitor",
    "intel_analyst",
)


class SurveillanceState(TypedDict, total=False):
    query_text: str

    # Created by the orchestrator.
    required_agents: list[str]
    agent_task_map: dict[str, str]
    current_agent_index: int
    direct_response: str | None

    # populated by the specialists.
    security_monitor_insights: dict[str, Any] | None
    broker_monitor_insights: dict[str, Any] | None
    risk_monitor_insights: dict[str, Any] | None
    intel_analyst_insights: dict[str, Any] | None

    # populated by the synthesizer.
    final_response: str | None


#define helper functions
def parse_json_object(text: str) -> dict:
    """Parse a JSON object from plain text or a Markdown code fence."""
    candidate = text.strip()

    if candidate.startswith("```"):
        first_newline = candidate.find("\n")
        candidate = candidate[first_newline + 1:]
        if candidate.endswith("```"):
            candidate = candidate[:-3].strip()

    try:
        value = json.loads(candidate)
    except json.JSONDecodeError:
        start = candidate.find("{")
        if start < 0:
            raise
        value, _ = json.JSONDecoder().raw_decode(candidate[start:])

    if not isinstance(value, dict):
        raise ValueError("Routing response must be a JSON object.")

    return value


def validate_route(
    payload: dict,
    fallback_query: str,
) -> tuple[list[str], dict[str, str]]:
    """Discard unknown, duplicate, or malformed agent tasks."""
    required_agents = []
    task_map = {}

    for item in payload.get("agent_tasks", []):
        if not isinstance(item, dict):
            continue

        name = item.get("agent")
        task = item.get("task")

        if name not in SPECIALISTS or name in task_map:
            continue

        required_agents.append(name)
        task_map[name] = (
            task.strip()
            if isinstance(task, str) and task.strip()
            else fallback_query
        )

    return required_agents, task_map


async def stream_with_events(agent: Agent, task: str) -> str:
    """Stream a Strands agent, emitting lifecycle and tool events to the graph."""
    writer = get_stream_writer()
    name = agent.name

    writer({"type": "agent_execution", "agent": name, "status": "started"})

    chunks, seen_tool_ids = [], set()
    async for event in agent.stream_async(task):
        if "data" in event:
            chunks.append(event["data"])

        tool_use = event.get("current_tool_use") or {}
        tool_id = tool_use.get("toolUseId")
        tool_name = tool_use.get("name")
        if tool_name and tool_id and tool_id not in seen_tool_ids:
            writer({
                "type": "tool",
                "agent": name,
                "name": tool_name,
                "tool_use_id": tool_id,
                "status": "started",
            })
            seen_tool_ids.add(tool_id)

    writer({"type": "agent_execution", "agent": name, "status": "completed"})
    return "".join(chunks)


### Orchestrator

async def orchestrator_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()

    orchestrator = Agent(
        name="orchestrator",
        model=fast_model,
        system_prompt=ORCHESTRATOR_AGENT_PROMPT,
        tools=[],
        callback_handler=None,
    )

    writer({
        "type": "agent_execution",
        "agent": "orchestrator",
        "status": "started",
    })

    chunks = []
    async for event in orchestrator.stream_async(state["query_text"]):
        if "data" in event:
            chunks.append(event["data"])

    response = "".join(chunks)

    try:
        payload = parse_json_object(response)
        required_agents, task_map = validate_route(
            payload,
            state["query_text"],
        )
        direct_response = payload.get("direct_response")

        if not required_agents and not direct_response:
            direct_response = (
                "Please provide a security symbol, date, "
                "and analysis objective."
            )
    except (json.JSONDecodeError, ValueError):
        required_agents, task_map = [], {}
        direct_response = response.strip() or "Unable to route the request."

    writer({"type": "route", "agents": required_agents})

    if not required_agents:
        writer({
            "type": "text",
            "agent": "orchestrator",
            "content": direct_response,
        })

    writer({
        "type": "agent_execution",
        "agent": "orchestrator",
        "status": "completed",
    })

    return {
        "required_agents": required_agents,
        "agent_task_map": task_map,
        "current_agent_index": 0,
        "direct_response": direct_response,
        "security_monitor_insights": None,
        "broker_monitor_insights": None,
        "risk_monitor_insights": None,
        "intel_analyst_insights": None,
        "final_response": None,
    }


### Security Monitor

async def security_monitor_node(state: SurveillanceState) -> dict:
    agent = Agent(
        name="security_monitor",
        model=analyst_model,
        system_prompt=SECURITY_MONITOR_AGENT_PROMPT,
        tools=[get_report_list, get_report_schema, run_report],
        callback_handler=None,
    )
    task = (state.get("agent_task_map") or {}).get("security_monitor", state["query_text"])
    result = await stream_with_events(agent, task)
    return {
        "security_monitor_insights": {"task": task, "business_insights": result},
        "current_agent_index": state.get("current_agent_index", 0) + 1,
    }


### Broker Monitor

async def broker_monitor_node(state: SurveillanceState) -> dict:
    agent = Agent(
        name="broker_monitor",
        model=analyst_model,
        system_prompt=BROKER_MONITOR_AGENT_PROMPT,
        tools=[get_report_list, get_report_schema, run_report],
        callback_handler=None,
    )
    task = (state.get("agent_task_map") or {}).get("broker_monitor", state["query_text"])
    result = await stream_with_events(agent, task)
    return {
        "broker_monitor_insights": {"task": task, "business_insights": result},
        "current_agent_index": state.get("current_agent_index", 0) + 1,
    }


### Risk Monitor

async def risk_monitor_node(state: SurveillanceState) -> dict:
    agent = Agent(
        name="risk_monitor",
        model=analyst_model,
        system_prompt=RISK_MONITOR_AGENT_PROMPT,
        tools=[get_report_list, get_report_schema, run_report],
        callback_handler=None,
    )
    task = (state.get("agent_task_map") or {}).get("risk_monitor", state["query_text"])
    result = await stream_with_events(agent, task)
    return {
        "risk_monitor_insights": {"task": task, "business_insights": result},
        "current_agent_index": state.get("current_agent_index", 0) + 1,
    }


### Intelligence Analyst

async def intel_analyst_node(state: SurveillanceState) -> dict:
    agent = Agent(
        name="intel_analyst",
        model=analyst_model,
        system_prompt=INTEL_ANALYST_AGENT_PROMPT,
        tools=[http_request],
        callback_handler=None,
    )
    task = (state.get("agent_task_map") or {}).get("intel_analyst", state["query_text"])
    result = await stream_with_events(agent, task)
    return {
        "intel_analyst_insights": {"task": task, "business_insights": result},
        "current_agent_index": state.get("current_agent_index", 0) + 1,
    }


### Synthesizer

SYNTHESIZER_PROMPT = (
    "Combine specialist findings into a concise market-surveillance summary. "
    "Separate observations from limitations, cite supplied dates and metrics, "
    "and do not add unsupported facts."
)


async def synthesizer_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()

    synthesizer = Agent(
        name="synthesizer",
        model=fast_model,
        system_prompt=SYNTHESIZER_PROMPT,
        tools=[],
        callback_handler=None,
    )

    findings = []
    insight_fields = (
        ("SECURITY MONITOR", "security_monitor_insights"),
        ("BROKER MONITOR", "broker_monitor_insights"),
        ("RISK MONITOR", "risk_monitor_insights"),
        ("INTELLIGENCE ANALYST", "intel_analyst_insights"),
    )

    for label, field in insight_fields:
        if insight := state.get(field):
            findings.append(f"{label}:\n{insight['business_insights']}")

    findings_text = "\n\n".join(findings)
    prompt = (
        f"Original question:\n{state['query_text']}\n\n"
        f"Specialist findings:\n{findings_text}"
    )

    chunks = []
    writer({"type": "agent_execution", "agent": "synthesizer", "status": "started"})

    async for event in synthesizer.stream_async(prompt):
        if "data" in event:
            chunks.append(event["data"])
            writer({"type": "text", "agent": "synthesizer", "content": event["data"]})

    writer({"type": "agent_execution", "agent": "synthesizer", "status": "completed"})
    return {"final_response": "".join(chunks)}


### Routing and Compilation

def route_next(state: SurveillanceState) -> str:
    required_agents = state.get("required_agents") or []
    index = state.get("current_agent_index", 0)

    if not required_agents:
        return END
    if index < len(required_agents):
        return required_agents[index]
    return "synthesizer"


workflow = StateGraph(SurveillanceState)

workflow.add_node("orchestrator", orchestrator_node)
workflow.add_node("security_monitor", security_monitor_node)
workflow.add_node("broker_monitor", broker_monitor_node)
workflow.add_node("risk_monitor", risk_monitor_node)
workflow.add_node("intel_analyst", intel_analyst_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.add_edge(START, "orchestrator")

all_targets = {
    **{name: name for name in SPECIALISTS},
    "synthesizer": "synthesizer",
    END: END,
}

workflow.add_conditional_edges("orchestrator", route_next, all_targets)
workflow.add_conditional_edges("security_monitor", route_next, all_targets)
workflow.add_conditional_edges("broker_monitor", route_next, all_targets)
workflow.add_conditional_edges("risk_monitor", route_next, all_targets)
workflow.add_conditional_edges("intel_analyst", route_next, all_targets)
workflow.add_edge("synthesizer", END)

#Use AgentCore Memory for persistent sessions
checkpointer = AgentCoreMemorySaver(MEMORY_ID, region_name=region)
full_graph = workflow.compile(checkpointer=checkpointer)

Test the agent by invoking the full graph

In [ ]:
query = (
      "Summarize AAPL intraday price and volume on March 15, compare its "
      "March 14 and March 15 daily performance, and identify broker market "
      "share and anomaly flags on March 15, 2024."
  )

async for event in full_graph.astream(
    {"query_text": query},
    config={"configurable": {"thread_id": f"workshop-{uuid4()}", "actor_id": "workshop-user"}},
    stream_mode="custom",
):
    if event.get("type") == "route":
        print(f"\nRoute: {event['agents']}")
    elif event.get("type") == "agent_execution":
        print(f"\n[{event['status']}] {event['agent']}")
    elif event.get("type") == "tool":
        print(f"  [tool:{event['agent']}] {event['name']}")
    elif event.get("type") == "text":
        print(event["content"], end="", flush=True)

## 8. The AgentCore Entry Point

Our agent runs locally, now we need to deploy it. AgentCore Runtime provides a serverless runtime designed speicifically for agents, with scalability, fast cold starts and per-session isolation.

To run on AgentCore we wrap the workflow in a `BedrockAgentCoreApp`. The `@app.entrypoint` decorator marks the async function AgentCore invokes for each request: it receives the JSON payload and yields streaming chunks. The real entry point is [`api.py`](../api.py) in the project root - open it to see the production version (logging, error handling, `session_id` propagation). It is intentionally tiny:

```python
# api.py (simplified)
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from src.agents import Workflow

app = BedrockAgentCoreApp()
workflow = Workflow()

@app.entrypoint
async def market_surveillance_workflow(payload):
    prompt = payload.get("prompt")
    session_id = payload.get("session_id", "default-session")
    async for chunk in workflow.stream_query(session_id=session_id, prompt=prompt):
        yield chunk

if __name__ == "__main__":
    app.run()
```

**What happens behind the scenes?** `BedrockAgentCoreApp` runs a small HTTP server on port 8080, exposes the `/invocations` endpoint AgentCore calls for each request, and adds a `/ping` health check - so you write an async function, not a web service. The `Dockerfile` packages this with all dependencies into an ARM64 container image; AgentCore handles orchestration, scaling, and networking.

## 9. Deploy to AgentCore Runtime

We use the `bedrock-agentcore-starter-toolkit` to deploy in one step. It:

1. Uploads your source to S3
2. Triggers an AWS CodeBuild job that compiles an ARM64 container image
3. Pushes the image to ECR
4. Creates (or updates) an AgentCore Runtime endpoint

> **Expected time: 5-10 minutes** for the first build. Subsequent runs reuse the existing runtime.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

AGENT_NAME = "market_surveillance_workflow"

agentcore_runtime = Runtime()

agentcore_runtime.configure(
    entrypoint="api.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=AGENT_NAME,
)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "api.py").exists() else Path.cwd().parent

# Re-running this cell after the Cleanup section deleted the local
# .bedrock_agentcore.yaml, so launch() starts fresh and creates a new runtime.
# If a runtime of this name still exists in the account (e.g. you re-run without
# cleaning up), auto_update_on_conflict=True updates it in place instead of
# raising ConflictException.
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "BEDROCK_AGENTCORE_MEMORY_ID": MEMORY_ID,
        "BEDROCK_AGENTCORE_MEMORY_NAME": MEMORY_NAME,
    },
)

print(f"Agent ARN : {launch_result.agent_arn}")
print(f"Agent ID  : {launch_result.agent_id}")
print(f"ECR URI   : {launch_result.ecr_uri}")


## 10. Invoke the Deployed Agent

AgentCore returns a **server-sent events (SSE) stream**. The helper below parses each chunk and prints it as it arrives, tagging agent start/stop and tool calls. (The Streamlit client in `app.py` consumes the same stream to drive its UI.)

In [ ]:
import json


def _parse_sse_line(line):
    """Parse one SSE line into a chunk dict, handling AgentCore's double-wrapped
    `data: "data: {...}"` framing. Returns None for keepalives / blank lines."""
    line = line.strip()
    if not line.startswith("data: "):
        return None
    inner = line[6:]
    try:
        unwrapped = json.loads(inner)  # strips the outer quoting
        if isinstance(unwrapped, str):
            unwrapped = unwrapped.strip()
            if unwrapped.startswith("data: "):
                unwrapped = unwrapped[6:].strip()
            return json.loads(unwrapped)
        if isinstance(unwrapped, dict):
            return unwrapped
    except (json.JSONDecodeError, TypeError):
        pass
    return None


def stream_response(boto3_response):
    """Stream an AgentCore SSE response and print each chunk as it arrives."""
    for raw in boto3_response["response"].iter_lines():
        line = raw.decode("utf-8") if isinstance(raw, bytes) else raw
        chunk = _parse_sse_line(line)
        if not isinstance(chunk, dict):
            continue
        ctype, agent = chunk.get("type", "text"), chunk.get("agent", "")
        if ctype == "agent_execution":
            mark = ">>" if chunk.get("status") == "started" else "ok"
            print(f"\n[{mark}] {agent}")
        elif ctype == "tool":
            print(f"\n   (tool) {chunk.get('name', '')}")
        elif ctype in ("text", "reasoningText"):
            content = chunk.get("content", "")
            if isinstance(content, list):
                content = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
            print(content, end="", flush=True)


print("Helper loaded: stream_response()")

In [ ]:
from botocore.config import Config

agentcore_client = boto3.client(
    "bedrock-agentcore",
    region_name=region,
    config=Config(read_timeout=600)  # 10 minutes
)


### Query 1 - Single-security intraday analysis

In [ ]:
agent_arn = launch_result.agent_arn

stream_response(agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "What caused the price for AAPL to increase at 11:00 AM on March 15, 2024?",
        "session_id": "workshop-session-001",
    }),
))

### Query 2 - Cross-security broker summary

In [ ]:
stream_response(agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "Summarise broker activity across all securities on March 15, 2024.",
        "session_id": "workshop-session-002",
    }),
))

### Query 3 - Conversational follow-up

Using the same `session_id` enables conversational memory. The orchestrator sees the previous query context and can answer follow-up questions without re-running the full analysis.

In [ ]:
stream_response(agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "Which broker had the highest risk score?",
        "session_id": "workshop-session-002",  # same session as Query 2
    }),
))

---
## 11. Cleanup

> **Run this section when you are finished** to avoid ongoing AWS charges.

The cells below delete the AgentCore Runtime and ECR repository. These are the resources that incur costs.

The cleanup cell also removes the local `.bedrock_agentcore.yaml` state file, so re-running the deploy section later starts from a clean slate.

**Not deleted automatically** (use the AWS console if you no longer need them):
- IAM execution role
- CodeBuild project
- S3 source bucket

In [ ]:
import yaml

# Read IDs from toolkit config
with open(PROJECT_ROOT / ".bedrock_agentcore.yaml") as f:
    toolkit_config = yaml.safe_load(f)

agent_config = toolkit_config["agents"][AGENT_NAME]
agent_arn = agent_config["bedrock_agentcore"]["agent_arn"]
agent_id = agent_config["bedrock_agentcore"]["agent_id"]
ecr_repo_name = agent_config["aws"]["ecr_repository"].split("/")[-1]

print("About to delete:")
print(f"  AgentCore Runtime : {agent_id}")
print(f"  ECR repository    : {ecr_repo_name}")
print(f"  AgentCore Memory  : {MEMORY_ID}")

In [ ]:
# Delete AgentCore Runtime
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)
agentcore_control.delete_agent_runtime(agentRuntimeId=agent_id)
print(f"Deleted AgentCore Runtime: {agent_id}")

# Delete ECR repository (force=True removes all images)
ecr_client = boto3.client("ecr", region_name=region)
ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
print(f"Deleted ECR repository: {ecr_repo_name}")

# Delete the AgentCore Memory created in Part 1 (the deployment reused this same
# resource, so there is only one memory to remove).
try:
    agentcore_control.delete_memory(memoryId=MEMORY_ID)
    print(f"Deleted AgentCore Memory: {MEMORY_ID}")
except Exception as e:
    print(f"Skipped memory {MEMORY_ID}: {type(e).__name__}")

# Remove local toolkit state so a later re-run of the deploy cell starts fresh
# (no stale agent_id pointing at the runtime we just deleted).
stale_cfg = PROJECT_ROOT / ".bedrock_agentcore.yaml"
if stale_cfg.exists():
    stale_cfg.unlink()
    print(f"Removed local config: {stale_cfg.name}")

print("\nCleanup complete.")

---
## What You Built

| Step | What happened |
|------|--------------|
| **Part 1** - Tools | Defined `@tool` functions wrapping mock market data |
| **Part 1** - Agent | Created a Strands agent that chains tools to answer questions |
| **Part 1** - Streaming | Used `stream_async` to observe agent events in real time |
| **Part 1** - Orchestrator | Added a routing agent that parses queries into tasks |
| **Part 1** - LangGraph | Wired orchestrator -> analyst -> synthesizer into a state graph with `get_stream_writer()` bridging |
| **Part 2** - Full system | Imported the six-agent production workflow |
| **Part 2** - Deploy | One-command deployment to AgentCore Runtime via starter toolkit |
| **Part 2** - Invoke | Streamed multi-agent responses over SSE from a managed endpoint |

## Next Steps

- **Add a custom tool** - create a new `@tool` in `src/tools/` and register it with the relevant specialist agent
- **Connect real data** - replace `MOCK_DATA` with actual database queries
- **Enable authentication** - pass `authorizer_configuration` to `configure()` for JWT or Cognito-based access
- **Add a Streamlit UI** - run `streamlit run app.py` to use the included chat interface
- **Monitor with CloudWatch** - AgentCore emits metrics and logs; create a dashboard to track latency and errors